# ResqAI — Hybrid Medical Fine-tuning Pipeline

**Model:** `google/gemma-4-e2b-it`  
**Framework:** Unsloth + QLoRA + 4-bit quantization  
**Target:** Kaggle T4 GPU (16GB VRAM)  

## Dataset Composition
- 50% Synthetic emergency triage data (`resqai_dataset.json`)
- 35% Converted MedQA reasoning data (`phrases_no_exclude_*.jsonl`)
- 15% Augmented protocol-based examples

## Pipeline Sections
1. Install dependencies
2. Imports & HuggingFace login
3. Preprocess MedQA dataset
4. Build hybrid dataset
5. Load model (4-bit)
6. Apply LoRA
7. Format dataset with chat template
8. Configure SFTTrainer
9. Train
10. Save model
11. Export GGUF Q4_K_M
12. Push to HuggingFace
13. Inference tests (5 scenarios)

## Section 1 — Install Dependencies

In [ ]:
# Install all required packages for Kaggle T4 environment
# unsloth[kaggle-new] is the Kaggle-specific variant for CUDA 11.8 compatibility
!pip install "unsloth[kaggle-new]" xformers trl peft accelerate bitsandbytes datasets transformers huggingface_hub -q
print("All dependencies installed.")

## Section 2 — Imports & HuggingFace Login

In [ ]:
import os, json, re, random, hashlib, logging
from pathlib import Path
from collections import Counter
from typing import Optional

import torch
from datasets import Dataset
from transformers import TrainingArguments
from trl import SFTTrainer, train_on_responses_only
from unsloth import FastLanguageModel
from huggingface_hub import login

# HuggingFace login via Kaggle secrets
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")
login(token=HF_TOKEN)

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
print(f"GPU             : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")
print("HuggingFace login successful.")

## Section 3 — Preprocess MedQA Dataset

Transforms raw MCQ entries into ResqAI conversational triage format.
No exam-style phrasing is preserved — each entry becomes a realistic patient/bystander message.

In [ ]:
SYSTEM_PROMPT = (
    "You are ResqAI, an emergency first-aid assistant. "
    "Always respond ONLY with structured JSON emergency triage output. "
    "Prioritize immediate stabilization and medically safe first-aid guidance. "
    "Be concise, calm, and decisive. "
    "Respond in the same language as the user."
)

# Keyword -> (condition_slug, severity, call_ambulance)
EMERGENCY_MAP = {
    "cardiac arrest":           ("cardiac_arrest",        "critical", True),
    "ventricular fibrillation": ("cardiac_arrest",        "critical", True),
    "myocardial infarction":    ("chest_pain",            "critical", True),
    "heart attack":             ("chest_pain",            "critical", True),
    "unstable angina":          ("chest_pain",            "high",     True),
    "chest pain":               ("chest_pain",            "high",     True),
    "aortic dissection":        ("chest_pain",            "critical", True),
    "respiratory failure":      ("breathing_difficulty",  "critical", True),
    "pulmonary embolism":       ("breathing_difficulty",  "critical", True),
    "shortness of breath":      ("breathing_difficulty",  "high",     True),
    "choking":                  ("choking_adult",         "critical", True),
    "anaphylaxis":              ("anaphylaxis",           "critical", True),
    "anaphylactic":             ("anaphylaxis",           "critical", True),
    "stroke":                   ("stroke",                "critical", True),
    "cerebrovascular":          ("stroke",                "critical", True),
    "seizure":                  ("seizure",               "high",     True),
    "status epilepticus":       ("seizure",               "critical", True),
    "loss of consciousness":    ("head_injury",           "high",     True),
    "unconscious":              ("head_injury",           "high",     True),
    "head injury":              ("head_injury",           "high",     True),
    "traumatic brain":          ("head_injury",           "critical", True),
    "spinal":                   ("spinal_injury",         "high",     True),
    "hemorrhage":               ("severe_bleeding",       "critical", True),
    "bleeding":                 ("severe_bleeding",       "high",     True),
    "thromboembolism":          ("severe_bleeding",       "high",     True),
    "disseminated intravascular":("severe_bleeding",      "critical", True),
    "fracture":                 ("fracture",              "moderate", False),
    "femur fracture":           ("fracture",              "high",     True),
    "burn":                     ("burn_second_degree",    "moderate", False),
    "overdose":                 ("opioid_overdose",       "critical", True),
    "poisoning":                ("opioid_overdose",       "high",     True),
    "hypoglycemia":             ("diabetic_hypoglycemia", "high",     True),
    "diabetic":                 ("diabetic_hypoglycemia", "moderate", False),
    "drowning":                 ("drowning",              "critical", True),
    "heat stroke":              ("heat_stroke",           "critical", True),
    "hyperthermia":             ("heat_stroke",           "high",     True),
    "sepsis":                   ("sepsis",                "critical", True),
    "septic shock":             ("sepsis",                "critical", True),
    "snakebite":                ("snakebite",             "high",     True),
    "electric shock":           ("electric_shock",        "high",     True),
    "nosebleed":                ("nosebleed",             "low",      False),
}
DEFAULT_CONDITION = ("medical_condition", "moderate", False)

NOISE_WORDS = {
    "following", "most likely", "best treatment", "patient", "physician",
    "presents", "history", "examination", "laboratory", "studies show",
    "which of", "correct next", "action", "cause", "pathogenesis",
    "precautions", "prevented", "account", "presentation", "diagnosis",
}

PANIC_TEMPLATES = [
    "{scenario} What do I do??",
    "HELP! {scenario}",
    "{scenario} Please help me!",
    "{scenario} I don't know what to do.",
    "Emergency! {scenario}",
    "{scenario}",
    "Someone help — {scenario}",
]

print("Constants defined.")

In [ ]:
# Per-condition first-aid step library (WHO/AHA/Red Cross 2020+)
STEP_LIBRARY = {
    "cardiac_arrest": [
        "Call 911 immediately.",
        "Begin CPR: 30 hard chest compressions, 2 inches deep.",
        "Give 2 rescue breaths after every 30 compressions.",
        "Use AED as soon as available — follow voice prompts.",
        "Deliver shock if advised, resume CPR immediately.",
        "Continue CPR until EMS arrives.",
    ],
    "chest_pain": [
        "Call 911 immediately.",
        "Have person sit or lie down comfortably.",
        "Loosen tight clothing around chest and neck.",
        "Give aspirin to chew if not allergic and available.",
        "Do not let them walk or exert themselves.",
        "Monitor breathing and consciousness until EMS arrives.",
    ],
    "breathing_difficulty": [
        "Call 911 immediately.",
        "Help person sit upright to ease breathing.",
        "Loosen any tight clothing around neck and chest.",
        "Keep them calm — anxiety worsens breathlessness.",
        "If prescribed inhaler available, assist with use.",
        "Begin CPR if breathing stops completely.",
    ],
    "choking_adult": [
        "Call 911 immediately.",
        "Stand behind victim, arms around waist.",
        "Place fist above navel, grasp with other hand.",
        "Give sharp inward-upward abdominal thrusts.",
        "Repeat until object expelled or victim collapses.",
        "Begin CPR if victim becomes unconscious.",
    ],
    "anaphylaxis": [
        "Call 911 immediately.",
        "Administer epinephrine auto-injector to outer thigh if available.",
        "Have person sit upright or lie with legs elevated.",
        "Loosen tight clothing around neck.",
        "Second epinephrine dose may be given after 5-15 minutes.",
        "Go to ER even if symptoms improve — biphasic reaction risk.",
    ],
    "stroke": [
        "Call 911 immediately — note the exact time symptoms started.",
        "Use FAST: Face drooping, Arm weakness, Speech difficulty.",
        "Have person sit or lie down safely.",
        "Do not give food, water, or medication.",
        "Do not drive — ambulance can begin treatment en route.",
        "Stay with them until EMS arrives.",
    ],
    "seizure": [
        "Call 911 if seizure lasts over 5 minutes.",
        "Clear area of hard or sharp objects.",
        "Do NOT restrain movements or put anything in mouth.",
        "Cushion head gently with something soft.",
        "Time the seizure from start.",
        "Turn person on side after shaking stops.",
    ],
    "head_injury": [
        "Call 911 if unconscious, confused, or vomiting.",
        "Keep person still — assume possible spinal injury.",
        "Do not give aspirin or ibuprofen.",
        "Apply ice pack wrapped in cloth to reduce swelling.",
        "Watch for: worsening headache, unequal pupils, confusion.",
        "Do not let them fall asleep without monitoring.",
    ],
    "spinal_injury": [
        "Call 911 immediately.",
        "Do NOT move the person — keep completely still.",
        "Support head and neck in neutral position.",
        "Keep person warm and calm.",
        "Stay with them until EMS arrives with spinal board.",
    ],
    "severe_bleeding": [
        "Call 911 immediately.",
        "Apply firm direct pressure with clean cloth.",
        "Do not remove cloth — add more on top if soaked.",
        "Elevate the injured limb above heart level.",
        "Apply tourniquet 2 inches above wound if limb bleeding severely.",
        "Keep person lying down and warm.",
    ],
    "burn_second_degree": [
        "Cool burn under cool running water for 20 minutes.",
        "Do not use ice, butter, or any home remedies.",
        "Do not pop blisters.",
        "Cover loosely with sterile non-stick dressing.",
        "Give over-the-counter pain reliever if available.",
        "Seek medical care if burn is larger than palm size.",
    ],
    "opioid_overdose": [
        "Call 911 immediately.",
        "Administer naloxone nasal spray if available.",
        "Place person on their side in recovery position.",
        "Tilt head back to open airway.",
        "Give rescue breaths if not breathing.",
        "Give second naloxone dose after 2-3 minutes if no response.",
    ],
    "diabetic_hypoglycemia": [
        "Give fast-acting sugar: glucose tablets, juice, or regular soda.",
        "Have person sit down safely.",
        "Wait 15 minutes and reassess symptoms.",
        "Give a snack with protein once improved.",
        "Call 911 if person loses consciousness.",
        "Do not give food or drink if unconscious.",
    ],
    "drowning": [
        "Call 911 immediately.",
        "Remove person from water carefully.",
        "Tilt head back, lift chin to open airway.",
        "Give 2 rescue breaths, then begin CPR.",
        "Continue CPR until EMS arrives.",
    ],
    "heat_stroke": [
        "Call 911 immediately.",
        "Move person to shade or air conditioning.",
        "Remove excess clothing.",
        "Apply cool wet cloths to neck, armpits, and groin.",
        "Fan vigorously to accelerate cooling.",
        "Give cool water only if person is conscious and can swallow.",
    ],
    "fracture": [
        "Keep person still and calm.",
        "Do not try to straighten the injured limb.",
        "Immobilize limb in position found with padding.",
        "Apply ice pack wrapped in cloth to reduce swelling.",
        "Seek medical care for X-ray evaluation.",
    ],
    "sepsis": [
        "Call 911 immediately.",
        "Keep person lying down and warm.",
        "Monitor breathing and consciousness.",
        "Do not give food or water.",
        "Note time symptoms started for EMS.",
    ],
    "snakebite": [
        "Call 911 immediately.",
        "Keep calm and still — movement spreads venom faster.",
        "Immobilize bitten limb below heart level.",
        "Remove rings, watches, and tight clothing near bite.",
        "Do NOT cut, suck, or apply tourniquet to bite.",
        "Do NOT apply ice.",
    ],
    "electric_shock": [
        "Do NOT touch victim until power source is confirmed off.",
        "Turn off power at main breaker immediately.",
        "Call 911 immediately.",
        "Check breathing and pulse once power is off.",
        "Begin CPR if not breathing.",
        "All electric shock victims need hospital evaluation.",
    ],
    "nosebleed": [
        "Sit upright and lean slightly forward.",
        "Pinch soft part of nose firmly for 10-15 minutes.",
        "Do not tilt head back.",
        "Apply cold compress to bridge of nose.",
        "Seek medical care if bleeding does not stop after 30 minutes.",
    ],
    "medical_condition": [
        "Keep person calm and comfortable.",
        "Monitor vital signs: breathing, pulse, consciousness.",
        "Do not give food, water, or medication without medical advice.",
        "Call a healthcare provider or urgent care line.",
        "Seek medical evaluation if symptoms worsen.",
    ],
}

WARN_MESSAGES = {
    "cardiac_arrest":        "Brain damage begins in 4 minutes without CPR.",
    "chest_pain":            "Possible heart attack — every minute of delay causes more damage.",
    "breathing_difficulty":  "Airway compromise can be fatal within minutes.",
    "choking_adult":         "Complete airway obstruction causes unconsciousness in minutes.",
    "anaphylaxis":           "Anaphylaxis without epinephrine can be fatal within minutes.",
    "stroke":                "Stroke treatment window is 3-4.5 hours — every minute counts.",
    "seizure":               "Seizure lasting over 5 minutes requires emergency intervention.",
    "head_injury":           "Brain bleed risk — do not give blood thinners or aspirin.",
    "spinal_injury":         "Moving a spinal injury victim can cause permanent paralysis.",
    "severe_bleeding":       "Severe arterial bleeding can cause death in minutes.",
    "opioid_overdose":       "Naloxone wears off before opioids — hospital evaluation required.",
    "drowning":              "Secondary drowning can occur hours later — ER evaluation required.",
    "heat_stroke":           "Core temperature must be reduced immediately.",
    "electric_shock":        "Do not approach until power is confirmed off.",
    "snakebite":             "Treat all snakebites as potentially venomous.",
    "sepsis":                "Septic shock can progress to organ failure rapidly.",
    "diabetic_hypoglycemia": "Severe hypoglycemia with altered consciousness needs IV glucose.",
}

NEXT_QUESTIONS = {
    "chest_pain":            "Is the person allergic to aspirin?",
    "breathing_difficulty":  "Is the person still breathing on their own?",
    "choking_adult":         "Can the person make any sound or cough at all?",
    "anaphylaxis":           "Do they have an epinephrine auto-injector available?",
    "stroke":                "When exactly did the symptoms start?",
    "seizure":               "Does the person have a known seizure disorder?",
    "head_injury":           "Did they lose consciousness, even briefly?",
    "spinal_injury":         "Can they feel and move their hands and feet?",
    "severe_bleeding":       "Is the blood spurting rhythmically or flowing steadily?",
    "burn_second_degree":    "How large is the burned area — bigger or smaller than the palm?",
    "opioid_overdose":       "Is naloxone (Narcan) available?",
    "diabetic_hypoglycemia": "Is the person conscious and able to swallow safely?",
    "fracture":              "Is the skin broken or is there any bone visible?",
    "snakebite":             "Can you describe the snake? Is swelling spreading?",
    "medical_condition":     "Are symptoms getting worse or staying the same?",
}

TIME_MAP = {
    "burn_second_degree": 20, "nosebleed": 15, "fracture": 15,
    "diabetic_hypoglycemia": 15, "heat_stroke": 20, "medical_condition": 30,
}

print("Step library and metadata defined.")

In [ ]:
def _seed(text): return int(hashlib.md5(text.encode()).hexdigest(), 16) % (2**31)

def classify_entry(question, answer, phrases):
    combined = " ".join([question.lower(), answer.lower()] + [p.lower() for p in phrases])
    for kw, mapping in EMERGENCY_MAP.items():
        if kw in combined:
            return mapping
    return DEFAULT_CONDITION

def extract_symptoms(question, phrases):
    out = []
    for p in phrases:
        pl = p.lower().strip()
        if len(pl) < 4: continue
        if any(n in pl for n in NOISE_WORDS): continue
        if any(c.isalpha() for c in pl): out.append(p.strip())
    return out[:6]

def extract_age_gender(question):
    m = re.search(r'(\d+)[- ]year[- ]old (man|woman|male|female|patient|child|boy|girl)', question, re.I)
    if m: return m.group(1), m.group(2)
    return None, None

def build_user_message(question, answer, phrases, condition_slug, rng):
    age, gender = extract_age_gender(question)
    symptoms = extract_symptoms(question, phrases)
    if age and gender:
        gw = "man" if gender.lower() in ("man","male") else "woman"
        prefix = f"My {age}-year-old {gw}"
    elif age:
        prefix = f"A {age}-year-old person"
    else:
        prefix = "Someone"
    if symptoms:
        scenario = f"{prefix} has {', '.join(symptoms[:3]).lower()}"
    else:
        first = question.split(".")[0]
        first = re.sub(r'(which of the following|what is the|most likely|best treatment|correct next action|most appropriate)\??',
                       '', first, flags=re.I).strip().rstrip(",")
        scenario = first
    template = rng.choice(PANIC_TEMPLATES)
    return re.sub(r'\s+', ' ', template.format(scenario=scenario)).strip()

def build_steps(condition_slug, severity, rng):
    steps = list(STEP_LIBRARY.get(condition_slug, STEP_LIBRARY["medical_condition"]))
    if severity in ("critical","high") and not steps[0].startswith("Call 911"):
        steps = ["Call 911 immediately."] + steps
    if len(steps) > 4:
        fixed, rest = steps[:2], steps[2:]
        rng.shuffle(rest)
        steps = fixed + rest
    n = rng.randint(4, min(6, len(steps)))
    return steps[:n]

def build_response(condition_slug, severity, call_ambulance, steps):
    time_val = 0 if severity in ("critical","high") else TIME_MAP.get(condition_slug, 20)
    warn = WARN_MESSAGES.get(condition_slug, "Seek emergency medical care immediately.") if severity in ("critical","high") else ""
    nq = "" if severity == "critical" else NEXT_QUESTIONS.get(condition_slug, "")
    return json.dumps({
        "severity": severity,
        "call_ambulance": call_ambulance,
        "steps": steps,
        "estimated_time_minutes": time_val,
        "condition": condition_slug,
        "warn_message": warn,
        "next_question": nq,
    }, ensure_ascii=False)

def convert_medqa_entry(entry):
    q = entry.get("question","")
    a = entry.get("answer","")
    phrases = entry.get("metamap_phrases",[])
    if not q or not a: return None
    rng = random.Random(_seed(q))
    cond, sev, call_amb = classify_entry(q, a, phrases)
    user_msg = build_user_message(q, a, phrases, cond, rng)
    steps = build_steps(cond, sev, rng)
    assistant_content = build_response(cond, sev, call_amb, steps)
    return {"messages": [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": user_msg},
        {"role": "assistant", "content": assistant_content},
    ]}

print("MedQA conversion functions defined.")

In [ ]:
# Process both MedQA files
# On Kaggle: files are uploaded as a dataset named 'resqai-dataset'
MEDQA_PATHS = [
    "/kaggle/input/resqai-dataset/phrases_no_exclude_train.jsonl",
    "/kaggle/input/resqai-dataset/phrases_no_exclude_test.jsonl",
]

medqa_entries = []
for path in MEDQA_PATHS:
    if not Path(path).exists():
        print(f"WARNING: {path} not found, skipping.")
        continue
    count = 0
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            try:
                entry = json.loads(line)
                result = convert_medqa_entry(entry)
                if result:
                    result["_source"] = "medqa"
                    medqa_entries.append(result)
                    count += 1
            except Exception as e:
                pass
    print(f"Converted {count} entries from {Path(path).name}")

print(f"\nTotal MedQA entries converted: {len(medqa_entries)}")

## Section 4 — Build Hybrid Dataset

Merges synthetic triage data + converted MedQA + augmented examples.
Applies typo, panic-style, and multilingual augmentation.

In [ ]:
# ── Validation ────────────────────────────────────────────────────────────────
REQUIRED_FIELDS = {"severity","call_ambulance","steps","estimated_time_minutes","condition","warn_message","next_question"}
VALID_SEVERITIES = {"critical","high","moderate","low"}
DANGEROUS_RE = re.compile(r'\d+\s*(mg|mcg|ml|units?|tablets?)\b', re.I)

def validate_entry(entry):
    msgs = entry.get("messages",[])
    if len(msgs) < 3: return False
    if [m["role"] for m in msgs] != ["system","user","assistant"]: return False
    try:
        p = json.loads(msgs[2]["content"])
    except: return False
    if REQUIRED_FIELDS - set(p.keys()): return False
    if p.get("severity") not in VALID_SEVERITIES: return False
    if not isinstance(p.get("call_ambulance"), bool): return False
    steps = p.get("steps",[])
    if not (4 <= len(steps) <= 8): return False
    if DANGEROUS_RE.search(json.dumps(p)): return False
    return True

# ── Augmentation helpers ──────────────────────────────────────────────────────
TYPOS = {"help":["hlep","helo"],"breathing":["breathng","brething"],
         "please":["plese","plz"],"emergency":["emergeny","emergancy"],
         "bleeding":["bleeing","bleding"],"collapsed":["colapsed","collpased"]}

MULTILINGUAL = [
    ("cardiac_arrest",   "मेरे पिताजी को दिल का दौरा पड़ा है, वो बेहोश हैं, सांस नहीं ले रहे, मैं क्या करूं?"),
    ("stroke",           "میرے والد کا چہرہ ایک طرف سے جھک گیا ہے، بات نہیں کر پا رہے، مدد کریں!"),
    ("choking_adult",    "¡Ayuda! Mi esposo se está ahogando con comida, no puede respirar, se está poniendo morado."),
    ("snakebite",        "सांप ने मुझे काट लिया है, मैं बाहर जंगल में हूं, पैर में सूजन आ रही है, मैं क्या करूं?"),
    ("severe_bleeding",  "بہت خون بہہ رہا ہے، ہاتھ کٹ گیا ہے، مدد کریں!"),
    ("burn_second_degree","मेरे बच्चे पर गर्म पानी गिर गया, उसकी उम्र 6 साल है, हाथ पर छाले पड़ गए हैं।"),
    ("cardiac_arrest",   "¡Ayuda! Mi esposo de 55 años se desmayó, no respira, no tiene pulso. Estamos en casa."),
    ("diabetic_hypoglycemia", "मेरे दोस्त को शुगर कम हो गई है, वो कांप रहा है और confused है, मैं क्या करूं?"),
]

def augment_typo(entry, rng):
    msgs = entry["messages"]
    words = msgs[1]["content"].split()
    new_words = [rng.choice(TYPOS[w.lower()]) if w.lower() in TYPOS and rng.random()<0.2 else w for w in words]
    new_msg = " ".join(new_words)
    if new_msg == msgs[1]["content"]: return None
    return {"messages": [msgs[0], {"role":"user","content":new_msg}, msgs[2]]}

def augment_panic(entry, rng):
    msgs = entry["messages"]
    prefixes = ["HELP!! ","PLEASE HELP ","EMERGENCY!! ","SOS!! "]
    new_msg = rng.choice(prefixes) + msgs[1]["content"].upper()
    return {"messages": [msgs[0], {"role":"user","content":new_msg}, msgs[2]]}

def make_multilingual_entry(condition_slug, user_msg):
    cond, sev, call_amb = EMERGENCY_MAP.get(condition_slug, DEFAULT_CONDITION)
    rng = random.Random(_seed(user_msg))
    steps = build_steps(cond, sev, rng)
    assistant_content = build_response(cond, sev, call_amb, steps)
    return {"messages": [
        {"role":"system",    "content": SYSTEM_PROMPT},
        {"role":"user",      "content": user_msg},
        {"role":"assistant", "content": assistant_content},
    ]}

print("Augmentation helpers defined.")

In [ ]:
rng = random.Random(42)

# ── Load synthetic triage data ────────────────────────────────────────────────
RESQAI_PATH = "/kaggle/input/resqai-dataset/resqai_dataset.json"
synthetic_entries = []
if Path(RESQAI_PATH).exists():
    with open(RESQAI_PATH, "r") as f:
        raw = json.load(f)
    for item in raw:
        convs = item.get("conversations",[])
        if len(convs) < 2: continue
        entry = {"messages": [
            {"role":"system",    "content": SYSTEM_PROMPT},
            {"role":"user",      "content": convs[0]["content"]},
            {"role":"assistant", "content": convs[1]["content"]},
        ], "_source": "synthetic"}
        if validate_entry(entry):
            synthetic_entries.append(entry)
    print(f"Loaded {len(synthetic_entries)} synthetic entries")
else:
    print(f"WARNING: {RESQAI_PATH} not found")

# ── Build multilingual entries ────────────────────────────────────────────────
multilingual_entries = []
for cond_slug, user_msg in MULTILINGUAL:
    entry = make_multilingual_entry(cond_slug, user_msg)
    entry["_source"] = "multilingual"
    if validate_entry(entry):
        multilingual_entries.append(entry)
print(f"Built {len(multilingual_entries)} multilingual entries")

# ── Augment base entries ──────────────────────────────────────────────────────
base_all = synthetic_entries + medqa_entries
augmented_entries = []
target_aug = max(int(len(base_all) * 0.18), 50)
attempts = 0
while len(augmented_entries) < target_aug and attempts < target_aug * 8:
    entry = rng.choice(base_all)
    strategy = rng.choice(["typo", "panic"])
    new_entry = augment_typo(entry, rng) if strategy == "typo" else augment_panic(entry, rng)
    if new_entry and validate_entry(new_entry):
        new_entry["_source"] = "augmented"
        augmented_entries.append(new_entry)
    attempts += 1
print(f"Generated {len(augmented_entries)} augmented entries")

# ── Combine and deduplicate ───────────────────────────────────────────────────
all_entries = synthetic_entries + medqa_entries + multilingual_entries + augmented_entries
seen_hashes = set()
unique_entries = []
for e in all_entries:
    h = hashlib.md5(e["messages"][1]["content"].encode()).hexdigest()
    if h not in seen_hashes:
        seen_hashes.add(h)
        unique_entries.append(e)

rng.shuffle(unique_entries)
print(f"\nTotal unique entries: {len(unique_entries)}")

# ── Train / eval split ────────────────────────────────────────────────────────
eval_size = max(int(len(unique_entries) * 0.1), 20)
eval_data  = unique_entries[:eval_size]
train_data = unique_entries[eval_size:]
print(f"Train: {len(train_data)} | Eval: {len(eval_data)}")

# ── Source distribution ───────────────────────────────────────────────────────
src_counts = Counter(e.get("_source","unknown") for e in unique_entries)
sev_counts = Counter()
for e in unique_entries:
    try: sev_counts[json.loads(e["messages"][2]["content"])["severity"]] += 1
    except: pass
print(f"Sources: {dict(src_counts)}")
print(f"Severity: {dict(sev_counts)}")

In [ ]:
# Save dataset statistics
cond_counts = Counter()
for e in unique_entries:
    try: cond_counts[json.loads(e["messages"][2]["content"])["condition"]] += 1
    except: pass

token_lengths = [int(sum(len(m["content"].split()) for m in e["messages"]) / 0.75) for e in unique_entries]

stats = {
    "total_entries": len(unique_entries),
    "train_entries": len(train_data),
    "eval_entries":  len(eval_data),
    "severity_distribution": dict(sev_counts),
    "condition_distribution": dict(cond_counts),
    "source_distribution": dict(src_counts),
    "token_length_stats": {
        "average": round(sum(token_lengths)/len(token_lengths), 1) if token_lengths else 0,
        "max": max(token_lengths) if token_lengths else 0,
        "min": min(token_lengths) if token_lengths else 0,
    }
}

Path("/kaggle/working").mkdir(exist_ok=True)
with open("/kaggle/working/dataset_statistics.json", "w") as f:
    json.dump(stats, f, indent=2)

print("Dataset statistics saved to /kaggle/working/dataset_statistics.json")
print(json.dumps(stats, indent=2))

## Section 5 — Load Model (4-bit QLoRA)

In [ ]:
max_seq_length = 2048
dtype = None        # Auto-detect: float16 on T4, bfloat16 on Ampere+
load_in_4bit = True # QLoRA: quantize base model to 4-bit to save VRAM

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "google/gemma-4-e2b-it",
    max_seq_length = max_seq_length,
    dtype          = dtype,
    load_in_4bit   = load_in_4bit,
    token          = HF_TOKEN,
)

print(f"Model loaded: google/gemma-4-e2b-it")
print(f"Parameters  : {model.num_parameters():,}")
if torch.cuda.is_available():
    used = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM used   : {used:.2f} GB / {total:.1f} GB")

## Section 6 — Apply LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r                        = 16,
    target_modules           = ["q_proj","k_proj","v_proj","o_proj",
                                 "gate_proj","up_proj","down_proj"],
    lora_alpha               = 16,
    lora_dropout             = 0,      # 0 = no dropout (Unsloth recommendation)
    bias                     = "none",
    use_gradient_checkpointing = "unsloth",  # Unsloth's optimized checkpointing
    random_state             = 42,
    use_rslora               = False,
    loftq_config             = None,
)

model.print_trainable_parameters()
if torch.cuda.is_available():
    used = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM after LoRA: {used:.2f} GB / {total:.1f} GB")

## Section 7 — Format Dataset with Chat Template

In [ ]:
def format_example(example):
    """Apply Gemma chat template to a messages list."""
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize           = False,
        add_generation_prompt = False,
    )
    return {"text": text}

# Strip internal _source key before creating HF Dataset
def clean(entries):
    return [{"messages": e["messages"]} for e in entries]

train_hf = Dataset.from_list(clean(train_data))
eval_hf  = Dataset.from_list(clean(eval_data))

train_formatted = train_hf.map(format_example, remove_columns=["messages"])
eval_formatted  = eval_hf.map(format_example,  remove_columns=["messages"])

print(f"Train formatted: {len(train_formatted)} examples")
print(f"Eval  formatted: {len(eval_formatted)} examples")
print("\nSample (first 400 chars):")
print(train_formatted[0]["text"][:400])

## Section 8 — Configure SFTTrainer with train_on_responses_only

In [ ]:
training_args = TrainingArguments(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,   # Effective batch = 2*4 = 8
    warmup_steps                = 10,
    num_train_epochs            = 3,
    learning_rate               = 2e-4,
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    logging_steps               = 10,
    optim                       = "adamw_8bit",
    weight_decay                = 0.01,
    lr_scheduler_type           = "cosine",
    seed                        = 42,
    output_dir                  = "/kaggle/working/resqai-checkpoints",
    save_strategy               = "epoch",
    evaluation_strategy         = "epoch",
    load_best_model_at_end      = True,
    report_to                   = "none",
)

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = train_formatted,
    eval_dataset       = eval_formatted,
    dataset_text_field = "text",
    max_seq_length     = max_seq_length,
    dataset_num_proc   = 2,
    packing            = False,
    args               = training_args,
)

# train_on_responses_only: only compute loss on assistant turns
# This prevents the model from learning to predict user/system tokens
# Gemma uses <start_of_turn>user and <start_of_turn>model markers
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part    = "<start_of_turn>model\n",
)

print("Trainer configured.")
print(f"Training on {len(train_formatted)} examples for {training_args.num_train_epochs} epochs")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Precision: {'bf16' if torch.cuda.is_bf16_supported() else 'fp16'}")

## Section 9 — Train

In [ ]:
print("Starting training...")
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

trainer_stats = trainer.train()

print("\n" + "="*55)
print("TRAINING COMPLETE")
print("="*55)
m = trainer_stats.metrics
print(f"  Runtime          : {m.get('train_runtime',0):.0f}s ({m.get('train_runtime',0)/60:.1f} min)")
print(f"  Final train loss : {m.get('train_loss',0):.4f}")
print(f"  Samples/sec      : {m.get('train_samples_per_second',0):.2f}")
if torch.cuda.is_available():
    peak = torch.cuda.max_memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  Peak VRAM        : {peak:.2f} GB / {total:.1f} GB")

## Section 10 — Save Model (LoRA Adapter + Merged)

In [ ]:
LORA_DIR  = "/kaggle/working/resqai-lora-adapter"
MERGED_DIR = "/kaggle/working/resqai-merged"

# Save LoRA adapter only (small, fast)
model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)
print(f"LoRA adapter saved to: {LORA_DIR}")

# Save merged model (LoRA weights merged into base — needed for GGUF export)
model.save_pretrained_merged(
    MERGED_DIR,
    tokenizer,
    save_method = "merged_16bit",
)
print(f"Merged model saved to: {MERGED_DIR}")

# List saved files
for d in [LORA_DIR, MERGED_DIR]:
    files = list(Path(d).iterdir()) if Path(d).exists() else []
    total_mb = sum(f.stat().st_size for f in files if f.is_file()) / 1e6
    print(f"  {d}: {len(files)} files, {total_mb:.0f} MB")

## Section 11 — Export GGUF Q4_K_M

Q4_K_M = 4-bit quantization with K-means clustering on mixed precision.
Provides ~95% of full-precision quality at ~25% of the file size.
Required for Ollama local deployment.

In [ ]:
GGUF_DIR = "/kaggle/working/resqai-gguf"

print("Exporting GGUF Q4_K_M...")
model.save_pretrained_gguf(
    GGUF_DIR,
    tokenizer,
    quantization_method = "q4_k_m",
)

print("GGUF export complete.")
for f in Path(GGUF_DIR).iterdir():
    size_gb = f.stat().st_size / 1e9
    print(f"  {f.name}: {size_gb:.2f} GB")

## Section 12 — Push to HuggingFace Hub

In [ ]:
# Replace with your HuggingFace username
HF_USERNAME = "YOUR_USERNAME"  # <-- change this
GGUF_REPO   = f"{HF_USERNAME}/resqai-gemma-e2b"
LORA_REPO   = f"{HF_USERNAME}/resqai-gemma-e2b-lora"

# Push GGUF model
print(f"Pushing GGUF to: {GGUF_REPO}")
model.push_to_hub_gguf(
    GGUF_REPO,
    tokenizer,
    quantization_method = "q4_k_m",
    token = HF_TOKEN,
)
print("GGUF pushed.")

# Push LoRA adapter
print(f"Pushing LoRA adapter to: {LORA_REPO}")
model.push_to_hub(LORA_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(LORA_REPO, token=HF_TOKEN)
print("LoRA adapter pushed.")

print(f"\nGGUF  : https://huggingface.co/{GGUF_REPO}")
print(f"LoRA  : https://huggingface.co/{LORA_REPO}")

## Section 13 — Inference Tests (5 Scenarios)

Validates the fine-tuned model produces valid JSON triage output.
Tests: cardiac arrest, stroke, burn, choking, multilingual snakebite.

In [ ]:
import re as _re

# ── Safety validator & fallback ───────────────────────────────────────────────
REQUIRED_FIELDS = {"severity","call_ambulance","steps","estimated_time_minutes",
                   "condition","warn_message","next_question"}
VALID_SEVERITIES = {"critical","high","moderate","low"}
DOSE_RE = _re.compile(r'\d+\s*(mg|mcg|ml|units?|tablets?)\b', _re.I)

FALLBACK = {
    "severity": "high", "call_ambulance": True,
    "steps": ["Call 911 immediately.","Keep the person calm and still.",
              "Monitor breathing and consciousness.",
              "Do not give food, water, or medication.",
              "Stay with the person until EMS arrives."],
    "estimated_time_minutes": 0, "condition": "unknown_emergency",
    "warn_message": "Unable to assess — call emergency services immediately.",
    "next_question": "Can you describe the symptoms in more detail?"
}

def extract_json(text):
    text = text.strip()
    if text.startswith("{"): return text
    m = _re.search(r'```(?:json)?\s*(\{.*?\})\s*```', text, _re.DOTALL)
    if m: return m.group(1)
    m = _re.search(r'\{[^{}]*"severity"[^{}]*\}', text, _re.DOTALL)
    if m: return m.group(0)
    return None

def safe_parse(text):
    raw = extract_json(text)
    if not raw: return FALLBACK, False, "no JSON found"
    try:
        p = json.loads(raw)
    except Exception as e:
        return FALLBACK, False, str(e)
    missing = REQUIRED_FIELDS - set(p.keys())
    if missing: return p, False, f"missing fields: {missing}"
    if p.get("severity") not in VALID_SEVERITIES: return p, False, "invalid severity"
    if not isinstance(p.get("call_ambulance"), bool): return p, False, "call_ambulance not bool"
    steps = p.get("steps",[])
    if not (4 <= len(steps) <= 8): return p, False, f"step count {len(steps)} out of range"
    if DOSE_RE.search(json.dumps(p)): return p, False, "medication dosage detected"
    return p, True, ""

print("Safety validator ready.")

In [ ]:
# Switch model to inference mode (disables dropout, enables KV cache)
FastLanguageModel.for_inference(model)

def run_inference(prompt, max_new_tokens=512):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda" if torch.cuda.is_available() else "cpu")

    with torch.no_grad():
        outputs = model.generate(
            input_ids        = inputs,
            max_new_tokens   = max_new_tokens,
            temperature      = 0.7,
            top_p            = 0.9,
            top_k            = 40,
            repetition_penalty = 1.1,
            do_sample        = True,
            use_cache        = True,
        )
    return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True).strip()

def print_result(tc_id, name, prompt, raw, parsed, valid, err):
    status = "PASS" if valid else "FAIL"
    bar = "=" * 55
    print(f"\n{bar}")
    print(f"{status} | {tc_id}: {name}")
    print(f"{bar}")
    print(f"Prompt : {prompt[:80]}...")
    if not valid:
        print(f"Error  : {err}")
        print(f"Raw    : {raw[:200]}")
    print(f"severity          : {parsed.get('severity')}")
    print(f"call_ambulance    : {parsed.get('call_ambulance')}")
    print(f"condition         : {parsed.get('condition')}")
    print(f"steps ({len(parsed.get('steps',[]))}):")
    for s in parsed.get('steps',[]):
        print(f"  - {s}")
    print(f"warn_message      : {parsed.get('warn_message','')}")
    print(f"next_question     : {parsed.get('next_question','')}")

print("Inference function ready.")

In [ ]:
TEST_CASES = [
    {
        "id": "TC-01", "name": "Cardiac Arrest + AED",
        "prompt": "45 year old man just collapsed in our office, no pulse, not breathing. There is an AED on the wall.",
        "expect_severity": "critical", "expect_ambulance": True,
        "must_contain": ["CPR", "AED", "911"],
    },
    {
        "id": "TC-02", "name": "Stroke — FAST Protocol",
        "prompt": "My 68-year-old father suddenly has face drooping on one side, his left arm is weak, and he can't speak properly. We're at home.",
        "expect_severity": "critical", "expect_ambulance": True,
        "must_contain": ["911", "time"],
    },
    {
        "id": "TC-03", "name": "Child Second-Degree Burn",
        "prompt": "My 6 year old spilled boiling water on her arm, there are blisters forming. I have a first aid kit.",
        "expect_severity": ["moderate","high"], "expect_ambulance": None,
        "must_contain": ["20", "cool"],
        "must_not_contain": ["ice", "butter", "pop"],
    },
    {
        "id": "TC-04", "name": "Adult Choking — Panicked Input",
        "prompt": "HELP!! Man at our restaurant is CHOKING cant breathe turning blue hes like 40s grabbing his throat PLEASE HELP",
        "expect_severity": "critical", "expect_ambulance": True,
        "must_contain": ["thrust", "911"],
    },
    {
        "id": "TC-05", "name": "Snakebite in Hindi (Multilingual)",
        "prompt": "\u0938\u093e\u0902\u092a \u0928\u0947 \u092e\u0941\u091d\u0947 \u0915\u093e\u091f \u0932\u093f\u092f\u093e \u0939\u0948, \u092e\u0948\u0902 \u092c\u093e\u0939\u0930 \u091c\u0902\u0917\u0932 \u092e\u0947\u0902 \u0939\u0942\u0902, \u092a\u0948\u0930 \u092e\u0947\u0902 \u0938\u0942\u091c\u0928 \u0906 \u0930\u0939\u0940 \u0939\u0948, \u092c\u0939\u0941\u0924 \u0926\u0930\u094d\u0926 \u0939\u0948, \u092e\u0948\u0902 \u0915\u094d\u092f\u093e \u0915\u0930\u0942\u0902?",
        "expect_severity": ["high","critical"], "expect_ambulance": True,
        "must_not_contain": ["tourniquet", "cut", "suck"],
    },
]

results = []
passed = 0

for tc in TEST_CASES:
    raw = run_inference(tc["prompt"])
    parsed, valid, err = safe_parse(raw)

    # Additional checks
    steps_text = " ".join(parsed.get("steps",[])).lower()
    full_text  = json.dumps(parsed).lower()

    checks = {"valid_json": valid}

    exp_sev = tc["expect_severity"]
    checks["severity"] = (parsed.get("severity") in exp_sev
                          if isinstance(exp_sev, list)
                          else parsed.get("severity") == exp_sev)

    if tc["expect_ambulance"] is not None:
        checks["call_ambulance"] = parsed.get("call_ambulance") == tc["expect_ambulance"]

    for kw in tc.get("must_contain", []):
        checks[f"has_{kw}"] = kw.lower() in steps_text

    for kw in tc.get("must_not_contain", []):
        checks[f"no_{kw}"] = kw.lower() not in full_text

    all_pass = all(checks.values())
    if all_pass: passed += 1

    print_result(tc["id"], tc["name"], tc["prompt"], raw, parsed, valid, err)
    print("Checks:", {k: ("PASS" if v else "FAIL") for k,v in checks.items()})
    results.append({"id": tc["id"], "name": tc["name"], "passed": all_pass, "checks": checks})

print(f"\n{'='*55}")
print(f"INFERENCE TEST SUMMARY: {passed}/{len(TEST_CASES)} passed")
print(f"{'='*55}")

# Save results
with open("/kaggle/working/inference_test_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("Results saved to /kaggle/working/inference_test_results.json")